In [1]:
import pandas as pd
import numpy as np

In [3]:
solv_feat = pd.read_csv("solvent_feature.csv")
df = pd.read_csv("Rf.csv")

In [14]:
solv_feat.head()

,SMILES,MolecularWeight,LogP,MolMR,TPSA,HBA,HBD,RotatableBonds,AromaticProportion,dipole,dielec
0,C1CCCCC1,84.162,2.34060,27.7020,0.00,0,0,0,1.000000,0.00,2.02
1,C1CCOC1,72.107,0.79680,20.0530,9.23,1,0,0,1.000000,1.63,7.58
2,CC#N,41.053,0.52988,11.2870,23.79,1,0,0,0.500000,3.50,37.50
3,CC(C)(C)O,74.123,0.77720,21.9718,20.23,1,1,0,1.000000,1.31,10.90
4,CC(C)=O,58.080,0.59530,16.3550,17.07,1,0,0,0.666667,2.85,20.70


In [20]:
def elu_encount_ratio(elu_smi):
    return (sum(df['eluent1_smiles']==elu_smi)+sum(df['eluent2_smiles']==elu_smi))/len(df)

In [11]:
polar = ['CC(C)=O','CC#N', 'CC(C)(C)O', 'CCO', 'CCOC(C)=O', 'CO', 'ClC(Cl)Cl', 'C[N+](=O)[O-]']
nonpolar = ['C1CCCCC1', 'CCC', 'CCCCC', 'CCCCCC', 'CCCCCCC', 'Cc1ccccc1', 'c1ccccc1']
neutral = [smi for smi in solv_feat['SMILES'] if smi not in polar and smi not in nonpolar]


In [21]:
elu_encount_ratio('CCOCC')

0.015893610120012974

In [38]:
def get_new_row(row, elu_smi,rf):
    sf = solv_feat[solv_feat['SMILES']==_elu].iloc[0]    
    _row = row.copy()
    _row['eluent1_smiles'] = elu_smi
    _row['ratio1'] = 1
    _row['ratio2'] = 0 
    _row['Rf'] = rf
    _row['MW1'] = sf['MolecularWeight'].item()
    _row['LogP1'] = sf['LogP']
    _row['MolMR1'] = sf['MolMR']
    _row['TPSA1'] = sf['TPSA']
    _row['HBA1'] = sf['HBA']
    _row['HBD1'] = sf['HBD']
    _row['RotB1'] = sf['RotatableBonds']
    _row['AromaticProportion1'] = sf['AromaticProportion']
    _row['dipole1'] = sf['dipole']
    _row['dielec1'] = sf['dielec']
    return _row

    
augment = []


for i in range(len(df)):
    row = df.iloc[i]
    smi = row['Smiles']
    elu1 = row['eluent1_smiles']
    elu2 = row['eluent2_smiles']
    if (elu1 in polar+neutral and row['ratio1'] >0.5) or (elu2 in polar+neutral and row['ratio2'] >0.5):
        for _elu in nonpolar:
            if np.random.rand() < elu_encount_ratio(_elu)*5:
                augment.append(get_new_row(row,_elu,0.05))
    elif (elu1 in nonpolar and row['ratio2'] <0.1 and row['Rf'] > 0.5) or (elu2 in nonpolar and row['ratio1'] <0.1 and row['Rf'] > 0.5):
        for _elu in polar:
            if np.random.rand() < elu_encount_ratio(_elu)*5:
                augment.append(get_new_row(row,_elu,0.95))            
    

In [39]:
augment_df = pd.DataFrame(augment)

In [40]:
augment_df

,Smiles,eluent1_smiles,eluent2_smiles,ratio1,ratio2,Rf,set,MW1,LogP1,MolMR1,...,MW2,LogP2,MolMR2,TPSA2,HBA2,HBD2,RotB2,AromaticProportion2,dipole2,dielec2
3,BrCc1ccc(-c2ccccc2)nc1,CCC,CO,1,0,0.05,train,44.097,1.4163,15.965,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
3,BrCc1ccc(-c2ccccc2)nc1,CCCCCC,CO,1,0,0.05,train,86.178,2.5866,29.816,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
4,COC(=O)c1ccc(NC(CCP(=O)(c2ccccc2)c2ccccc2)P(=O...,CCCCCC,CO,1,0,0.05,train,86.178,2.5866,29.816,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
5,COC(=O)c1ccc(NC(CCP(=O)(c2ccccc2)c2ccccc2)P(=O...,CCC,CO,1,0,0.05,train,44.097,1.4163,15.965,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
5,COC(=O)c1ccc(NC(CCP(=O)(c2ccccc2)c2ccccc2)P(=O...,CCCCCC,CO,1,0,0.05,train,86.178,2.5866,29.816,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24565,Clc1ccc(N2CCCCC2)cc1,CCOC(C)=O,CCOC(C)=O,1,0,0.95,train,88.106,0.5694,22.307,...,88.106,0.5694,22.3070,26.30,2,0,1,0.75,1.78,6.02
24566,Brc1cccc(N2CCCCC2)c1,CCOC(C)=O,CCOC(C)=O,1,0,0.95,train,88.106,0.5694,22.307,...,88.106,0.5694,22.3070,26.30,2,0,1,0.75,1.78,6.02
24567,Fc1ccccc1N1CCCCC1,CCOC(C)=O,CCOC(C)=O,1,0,0.95,train,88.106,0.5694,22.307,...,88.106,0.5694,22.3070,26.30,2,0,1,0.75,1.78,6.02
24611,N#Cc1c2c(cc(C(N)=O)c1-c1ccccc1)CCc1ccccc1-2,CCCCCC,CCCCCC,1,0,0.05,train,86.178,2.5866,29.816,...,86.178,2.5866,29.8160,0.00,0,0,3,1.00,0.00,1.90


In [41]:
pd.concat([df, augment_df]).to_csv("RF_aug.csv",index=False)

# larger ratio to 1

In [83]:
file="Rf_aug.csv"
df = pd.read_csv(file)

In [84]:
df_ratio_inverse = df[df['ratio1']-df['ratio2']<0]
df_ratio_inverse

,Smiles,eluent1_smiles,eluent2_smiles,ratio1,ratio2,Rf,set,MW1,LogP1,MolMR1,...,MW2,LogP2,MolMR2,TPSA2,HBA2,HBD2,RotB2,AromaticProportion2,dipole2,dielec2


In [77]:
col1 = [c for c in df_ratio_inverse.columns if '1' in c]
col2 = [c for c in df_ratio_inverse.columns if '2' in c]

In [78]:
df1 = df_ratio_inverse[col1].copy()
df2 = df_ratio_inverse[col2].copy()

In [79]:
df_ratio_inverse[col1]=df2
df_ratio_inverse[col2]=df1

/tmp/ipykernel_2410/1028141083.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ratio_inverse[col1]=df2
/tmp/ipykernel_2410/1028141083.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ratio_inverse[col2]=df1


In [80]:
df_ratio_inverse

,Smiles,eluent1_smiles,eluent2_smiles,ratio1,ratio2,Rf,set,MW1,LogP1,MolMR1,...,MW2,LogP2,MolMR2,TPSA2,HBA2,HBD2,RotB2,AromaticProportion2,dipole2,dielec2
269,OCCOCCNCCOCCO,CO,ClCCl,0.513367,0.486633,0.14,train,32.042,-0.3915,8.1428,...,84.933,1.4215,16.573,0.0,0,0,0,1.0,1.14,8.93
519,Cc1ccc(S(=O)(=O)Oc2ccc([S+]3c4ccccc4Sc4ccccc43...,CO,ClCCl,0.513367,0.486633,0.20,test,32.042,-0.3915,8.1428,...,84.933,1.4215,16.573,0.0,0,0,0,1.0,1.14,8.93
562,COC1=C(C2=C(OC)C(=O)c3c(-c4ccccc4)ccc4cccc2c34...,ClCCl,CCCCCC,0.671063,0.328937,0.30,train,84.933,1.4215,16.5730,...,86.178,2.5866,29.816,0.0,0,0,3,1.0,0.00,1.90
563,COc1cc(Br)c(SC(=S)N(C)Cc2ccccc2)cc1OC,ClCCl,CCCCCC,0.671063,0.328937,0.48,train,84.933,1.4215,16.5730,...,86.178,2.5866,29.816,0.0,0,0,3,1.0,0.00,1.90
568,COc1ccc(C(=O)C(c2ccc(N(C)C)cc2)C2C(C(=O)c3ccc(...,ClCCl,CCCCCC,0.504962,0.495038,0.15,train,84.933,1.4215,16.5730,...,86.178,2.5866,29.816,0.0,0,0,3,1.0,0.00,1.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19093,C=C[C@H](C)[C@H](O)CC(=O)N1CCN(c2ncccn2)CC1,CC(C)=O,CCCCCC,0.542507,0.457493,0.46,train,58.080,0.5953,16.3550,...,86.178,2.5866,29.816,0.0,0,0,3,1.0,0.00,1.90
19163,CCOP(=O)(CC(CCOC(=O)c1ccccn1)N=C(c1ccccc1)c1cc...,CC(C)=O,CCCCCC,0.542507,0.457493,0.30,train,58.080,0.5953,16.3550,...,86.178,2.5866,29.816,0.0,0,0,3,1.0,0.00,1.90
19164,CCOP(=O)(CC(CNC(=O)c1ccccc1)N=C(c1ccccc1)c1ccc...,CC(C)=O,CCCCCC,0.542507,0.457493,0.20,train,58.080,0.5953,16.3550,...,86.178,2.5866,29.816,0.0,0,0,3,1.0,0.00,1.90
19165,CCOP(=O)(CC(Cn1cc(C)nc1C)N=C(c1ccccc1)c1ccccc1...,CC(C)=O,CCCCCC,0.542507,0.457493,0.30,train,58.080,0.5953,16.3550,...,86.178,2.5866,29.816,0.0,0,0,3,1.0,0.00,1.90


In [81]:
df_new = pd.concat([df[df['ratio1']-df['ratio2']>=0],df_ratio_inverse])

In [82]:
df_new.to_csv(file,index=False)

# Major eluent filter

In [85]:
df = pd.read_csv("Rf.csv")

In [97]:
unique_values,counts  = np.unique([a+','+b for a,b in np.array(df[['eluent1_smiles','eluent2_smiles']])],return_counts=True,)

In [98]:
sorted_indices = np.argsort(-counts)  # -counts로 내림차순
sorted_values = unique_values[sorted_indices]
sorted_counts = counts[sorted_indices]

In [99]:
sorted_values

array(['CCCCCC,CCOC(C)=O', 'CCOC(C)=O,CCCCCC', 'CCCCCC,CCC', 'ClCCl,CO',
       'CCCCC,CCOC(C)=O', 'C1CCCCC1,CCOC(C)=O', 'CCCCCC,ClCCl',
       'CCCCC,CCOCC', 'CCCCCC,CC(C)=O', 'CCOC(C)=O,CCC',
       'ClCCl,CCOC(C)=O', 'CCCCCC,CCOCC', 'ClCCl,CCCCCC', 'ClCCl,CC(C)=O',
       'CCOC(C)=O,CCCCC', 'CCCCCCC,CCOC(C)=O', 'CCOC(C)=O,CO',
       'ClC(Cl)Cl,CO', 'CCCCC,CCC', 'ClCCl,CCC', 'CCOC(C)=O,CCCCCCC',
       'CO,ClCCl', 'CCOC(C)=O,C1CCCCC1', 'CCCCC,ClCCl',
       'Cc1ccccc1,CCCCCC', 'c1ccccc1,CCC', 'CCOCC,CCCCCC', 'CCOCC,CCCCC',
       'Cc1ccccc1,CCOC(C)=O', 'CC(C)=O,CCCCCC', 'ClCCl,CCCCC',
       'CCCCCC,CCCCCC', 'CCCCCC,Cc1ccccc1', 'CCCCC,CC(C)=O',
       'CC(C)=O,ClCCl', 'CO,CCN(CC)CC', 'CCOCC,ClCCl', 'c1ccccc1,CCOCC',
       'CCOC(C)=O,Cc1ccccc1', 'CCOC(C)=O,ClCCl', 'ClCCl,CCO', 'CCOCC,CCC',
       'C1CCCCC1,CCC', 'ClCCl,C1CCOC1', 'CCCCCC,COC(C)(C)C',
       'C1CCCCC1,CCOCC', 'CC(C)(C)O,CCOC(C)=O', 'CCCCC,CCCCCC',
       'ClCCl,Cc1ccccc1', 'ClCCl,CCOCC', 'CC(C)=O,CCC', 'CO,CCC',
     

In [107]:
mask = []
for i in range(len(df)):
    if df.iloc[i]['eluent1_smiles'] in ('CCCCCC','CCOC(C)=O') and df.iloc[i]['eluent2_smiles'] in ('CCCCCC','CCOC(C)=O'):
        mask.append(True)
    else:
        mask.append(False)
        

In [111]:
df_only_PE_EA = df[mask]

In [117]:
df_inverse = df_only_PE_EA[df_only_PE_EA['eluent1_smiles']=='CCOC(C)=O']

In [118]:
col1 = [c for c in df_only_PE_EA.columns if '1' in c]
col2 = [c for c in df_only_PE_EA.columns if '2' in c]

In [119]:
df1 = df_inverse[col1].copy()
df2 = df_inverse[col2].copy()

In [120]:
df_inverse[col1]=df2
df_inverse[col2]=df1

/tmp/ipykernel_2410/2573004578.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_inverse[col1]=df2
/tmp/ipykernel_2410/2573004578.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_inverse[col2]=df1


In [121]:
df_inverse

,Smiles,eluent1_smiles,eluent2_smiles,ratio1,ratio2,Rf,set,MW1,LogP1,MolMR1,...,MW2,LogP2,MolMR2,TPSA2,HBA2,HBD2,RotB2,AromaticProportion2,dipole2,dielec2
10715,CCCCCC(C=CCC(F)F)N1CCN(C2=Nc3ccccc3Sc3ccccc32)CC1,CCOC(C)=O,CCOC(C)=O,0.070000,0.930000,0.50,train,88.106,0.5694,22.307,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02
16601,C=C(Cc1ccccc1)C[P@]1(=O)C[C@](CC(=O)Oc2ccccc2)...,CCCCCC,CCOC(C)=O,0.271918,0.728082,0.60,train,86.178,2.5866,29.816,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02
16602,COC(=O)c1ccnc(-n2ccccc2=O)c1,CCCCCC,CCOC(C)=O,0.427572,0.572428,0.20,train,86.178,2.5866,29.816,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02
16603,COC1(c2ccccc2Br)CC=CC1,CCCCCC,CCOC(C)=O,0.242492,0.757508,0.50,train,86.178,2.5866,29.816,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02
16605,COC1CCOC(c2ccc(F)cc2)=N1,CCCCCC,CCOC(C)=O,0.199347,0.800653,0.35,train,86.178,2.5866,29.816,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24651,CC(C)(C)OC(=O)[C@@](C)(N)Cc1cn(C(=O)[O-])c2ccc...,CCCCCC,CCOC(C)=O,0.427572,0.572428,0.30,test,86.178,2.5866,29.816,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02
24652,NC(=O)[C@H]1O[C@@H](c2ccccc2)OC[C@@H]1O,CCCCCC,CCOC(C)=O,0.427572,0.572428,0.64,test,86.178,2.5866,29.816,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02
24654,COc1cc2c(cc1OC)C(=O)[C@@](CCc1ccccn1)(C(=O)OC(...,CCCCCC,CCOC(C)=O,0.427572,0.572428,0.30,test,86.178,2.5866,29.816,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02
24655,CC(=C=Cc1ccccc1)CP(=O)(c1ccc2sccc2c1)c1ccc2scc...,CCCCCC,CCOC(C)=O,0.427572,0.572428,0.20,test,86.178,2.5866,29.816,...,88.106,0.5694,22.307,26.3,2,0,1,0.75,1.78,6.02


In [124]:
pd.concat([ df_only_PE_EA[df_only_PE_EA['eluent1_smiles']!='CCOC(C)=O'], df_inverse]).to_csv("Rf_major.csv",index=False)